## PharmaNODE — FiLM Pipeline Tutorial

This tutorial demonstrates how to use `DrugStudyConfig` and `ClassicPKModel` to generate a **FiLM-compatible** dataset (paired visits per patient) and train a Latent ODE with FiLM conditioning.

### 1. Define the Configuration

In [ ]:
import os
import random
import torch
import numpy as np
import pandas as pd
from IPython.display import display
from lib.pk_drug import DrugStudyConfig, generate_virtual_cohort_film, save_cohort_splits
from lib.classic_pk import ClassicPKModel

config = DrugStudyConfig(
    exp_id="tutorial_film",
    population_params={"CL": 10.0, "Vc": 50.0, "ka": 1.2},
    ipv_omega={"CL": 0.25, "Vc": 0.2, "ka": 0.2},
    residual_prop_sd=0.10,
    residual_add_sd=0.01,
    dose_choices=[20.0, 30.0, 40.0, 50.0],
    dosing_interval_h=24.0,
    n_steady_state_cycles=4,
    observation_times=[0.0, 0.33, 0.67, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 9.0, 12.0, 24.0],
    sparse_times=[0.0, 1.0, 3.0], # FiLM default
    covariate_columns=["WT"]
)

### 2. Generate Paired Visits Dataset
`generate_virtual_cohort_film` automatically generates Visit 1 and Visit 2 for each patient, sharing their base PK parameters but applying different doses.

In [ ]:
def pk_factory(meta: dict):
    model = ClassicPKModel(n_compartments=1, absorption='oral', cov_data=meta, pop_params=config.population_params, ipv=config.ipv_omega)
    return model

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

cohort_df = generate_virtual_cohort_film(pk_factory, config, num_patients=200)

train_df, test_df = save_cohort_splits(cohort_df, config, test_size=0.2)
print("Saved train and test CSVs for FiLM in:", config.results_path)
display(cohort_df.head())

### 3. Build FiLM Dataloaders
We use `extract_gen_pk_film` and `TacroFilmDataset` to parse the paired visits.

In [ ]:
from lib.pk_drug import extract_gen_pk_film
from lib.read_tacro import TacroFilmDataset, collate_fn_tacro_film
from torch.utils.data import DataLoader

data_dict, scaler = extract_gen_pk_film(config)
print("Scaler max_out, boxcox lambda:", scaler)

dataset = TacroFilmDataset(data_dict)
loader = DataLoader(
    dataset,
    batch_size=min(16, len(dataset)),
    shuffle=False,
    collate_fn=lambda b: collate_fn_tacro_film(b, device=torch.device("cpu")),
)
batch = next(iter(loader))
print("Observed v1 shape:", batch["observed_data_v1"].shape)
print("Observed v2 shape:", batch["observed_data_v2"].shape)

### 4. Smoke-test FiLM Training
We run `run_models.py` with `--dataset PK_Generic` and `--use_film`.

In [ ]:
!python run_models.py --niters 5000 -n 50 -s 40 -l 10 --dataset PK_Generic --latent-ode --use_film \
  --noise-weight 0.01 --max-t 5. --seed 101 --experiment tutorial_film -b 32